# 5.4 章节实践

本节使用新输入完成四类考核。所有 NPU 结论必须来自本次真实设备运行；OM 文件存在、代码可导入或历史日志都不能直接写成 PASS。

## 一、客观题

1. `acl.op.set_model_dir` 的作用是：A. 加载整图模型；B. 注册单算子 OM 目录；C. 创建 Stream。
2. H2D 与 D2H 分别表示什么方向？为什么 `execute_v2`/`gemm_ex` 后必须先同步再 D2H？
3. 在 910B3/CANN 9.0 课程合同中，`SparseTensorDenseMatMul` 应标记为 AI Core 还是 AI CPU/tf_kernel？
4. 判断：`acl` 可导入且 OM 文件存在，可以直接写 `VALIDATION=PASS`。


## 二、简单编程题：GEMM

使用 `M=32、N=64、K=128` 的独立输入，通过 `acl.blas.gemm_ex` 完成真实设备计算。报告必须包含输出 shape、Device ID、实际调用入口、Event 平均时间、最大误差和状态。


In [ ]:
import numpy as np

M, N, K = 32, 64, 128
gemm_a = ((np.arange(M)[:, None] + np.arange(K)[None, :]) % 7).astype(np.float16)
gemm_b = ((np.arange(K)[:, None] * 2 + np.arange(N)[None, :]) % 9).astype(np.float16)
gemm_golden = (gemm_a.astype(np.float32) @ gemm_b.astype(np.float32)).astype(np.float16)
assert gemm_golden.shape == (M, N)
print("GEMM practice input:", gemm_a.shape, gemm_b.shape, gemm_golden.shape)


## 三、中等编程题：COO SpMV

使用 `M=64、N=128、NNZ=100`，先验证坐标 shape、范围和唯一性，再通过 `acl.op.execute_v2("SparseTensorDenseMatMul", ...)` 计算。与按 COO 直接累加的 NumPy reference 比较，并如实记录 AI CPU/tf_kernel 边界。


In [ ]:
import numpy as np

SM, SN, NNZ = 64, 128, 100
linear = np.arange(NNZ, dtype=np.int64) * (SM * SN // NNZ)
spmv_indices = np.stack([linear // SN, linear % SN], axis=1)
spmv_values = ((np.arange(NNZ) * 5 + 1) % 11 + 1).astype(np.float32)
spmv_x = ((np.arange(SN) * 3 + 1) % 13).astype(np.float32)

assert spmv_indices.shape == (NNZ, 2)
assert np.unique(spmv_indices, axis=0).shape[0] == NNZ
assert np.all((0 <= spmv_indices[:, 0]) & (spmv_indices[:, 0] < SM))
assert np.all((0 <= spmv_indices[:, 1]) & (spmv_indices[:, 1] < SN))
spmv_golden = np.zeros(SM, dtype=np.float32)
for (row, col), value in zip(spmv_indices, spmv_values):
    spmv_golden[row] += value * spmv_x[col]
print("SpMV practice input:", spmv_indices.shape, spmv_values.shape, spmv_x.shape)


## 四、困难编程题：统一 runner

把前两题的真实设备主链分别封装为 `run_gemm()` 和 `run_spmv()`，顺序执行并生成 `work/05.04_chapter_test/report.json`。要求：

- 每条路径只保留一层 `try/finally`，异常时仍逆序释放资源；
- 两份结果都包含 `actual_backend/device_id/max_abs_error/device_mean_ms/status/fallback`；
- SpMV 额外包含 `aicore_accelerated=false`；
- 只有两条真实设备路径都严格通过时，才输出 `VALIDATION=PASS`。

下面的验证器不执行算子，也不会替代 `run_gemm()`/`run_spmv()`；它只检查学生生成的真实结果。


In [ ]:
import json
from pathlib import Path


def write_validated_report(gemm_result, spmv_result):
    required = {
        "actual_backend", "device_id", "max_abs_error",
        "device_mean_ms", "status", "fallback",
    }
    for name, result in (("gemm", gemm_result), ("spmv", spmv_result)):
        missing = required - result.keys()
        if missing:
            raise AssertionError(f"{name} 缺少字段：{sorted(missing)}")
        assert result["status"] == "PASS" and result["fallback"] == 0
        assert result["device_mean_ms"] > 0 and result["max_abs_error"] >= 0
    assert spmv_result.get("aicore_accelerated") is False
    assert "AICPU" in spmv_result["actual_backend"]

    report = {
        "schema_version": 1,
        "gemm": gemm_result,
        "spmv": spmv_result,
        "validation": "PASS",
    }
    path = Path("work/01.04_chapter_test/report.json")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"VALIDATION=PASS report={path.resolve()}")
    return report

# 完成 run_gemm()/run_spmv() 后再执行：
# report = write_validated_report(gemm_result, spmv_result)


## 完成标准

客观题答案正确；简单题和中等题均来自真实 Device 0 路径并通过 NumPy 校验；困难题生成机器可读报告且只由严格验证器输出 `VALIDATION=PASS`。性能数值不设固定答案，不得把历史日志复制为本次结果。


In [ ]:
# 完成四类考核后按需执行；Notebook 不会自动展开答案。
!cat answer/05.04_chapter_test_answer.md
